# 02 - Data Preprocessing
Cleans the raw dataset and prepares train/test splits using the shared `backend.preprocessing` module (same code path used by the production pipeline).

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
from sklearn.model_selection import train_test_split
from backend.preprocessing import FEATURE_COLUMNS, TARGET, build_preprocessor

In [ ]:
df = pd.read_csv('../data/raw/yield_df.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
print('Missing values:', df.isna().sum().sum())
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
df = df.dropna().drop_duplicates()
df.shape

In [ ]:
X = df[FEATURE_COLUMNS]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

In [ ]:
# Example: build the tree-model preprocessor (no scaling) and fit on TRAIN ONLY
pre = build_preprocessor(scale_numeric=False)
pre.fit(X_train)
X_train_enc = pre.transform(X_train)
X_train_enc.shape

Preprocessing is fit on the training split only, then applied to the test split — this avoids data leakage. The same `ColumnTransformer` logic is reused at prediction time via the saved `Pipeline` objects, so training and inference preprocessing can never drift apart.